In [1]:
from data.data_pipeline import load_pamap2, filter_loco_pam, remove_zero_label_rows, incomplete_labeled_rows, interpolation_pam, acc_data_scaling_pam, mag_data_norm_pam, create_pamap_windows_from_split, mag_data_rotation_pam
from sklearn.preprocessing import StandardScaler, LabelEncoder
from torch.utils.data import DataLoader
from sklearn.model_selection import StratifiedGroupKFold
from pathlib import Path
import glob
import pandas as pd
import numpy as np
import torch
import sys
import gc
from MambaClassificationModel import MambaClassificationModel, HARMambaConfig
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from data.preprocessing import fit_labelencoder
from test_mamba import test_model, save_json 
from mamba_ssm.modules.mamba2 import Mamba2
import mamba_ssm.modules.mamba2 as mm
from datetime import datetime
import torch.nn.functional as F
import mamba_ssm
import torch.nn as nn
from tqdm import tqdm
from utils import set_seed
import argparse
import json
import os
from torch.profiler import profile, ProfilerActivity
from data.preprocessing import fit_labelencoder, Dataset_HAR
import time 
try:
    from mamba_ssm.ops.triton.layer_norm import RMSNorm, layer_norm_fn, rms_norm_fn
except ImportError:
    RMSNorm, layer_norm_fn, rms_norm_fn = None, None, None

print(f"Mamba version: {mamba_ssm.__version__}")
print(f"Path: {mm.__file__}")

ImportError: cannot import name 'create_pamap_windows_from_split' from 'data.data_pipeline' (/home/kmercad/mamba_har/SUPERVISED MAMBA/Mamba_Baseline_PAM/data/data_pipeline.py)

In [2]:
!hostname
import sys, os, torch
print("python:", sys.executable)
print("cuda visible devices:", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("device count:", torch.cuda.device_count())
!nvidia-smi

gpu001.orc.gmu.edu
python: /home/kmercad/myenv/bin/python
cuda visible devices: 0
torch: 2.5.1+cu121
cuda available: True
device count: 1
Mon Jun  8 11:59:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 560.35.05              Driver Version: 560.35.05      CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          On  |   00000000:C1:00.0 Off |                    0 |
| N/A   41C    P0            270W /  500W |    5473MiB /  81920MiB |     86%      Default |
| 

In [4]:
def data_split_PAM(fold_id: int = 1) -> tuple[list, list, list]:
    '''
    Predefined subject split for PAMAP2 dataset.
    For a given fold_id, the corresponding subjects are held out as test.
    
    fold_id 1 → test: 101, 102 | val: 103, 104 | train: 105, 106, 107, 108
    fold_id 2 → test: 107, 108 | val: 101, 102 | train: 103, 104, 105, 106
    fold_id 3 → test: 105, 106 | val: 107, 108 | train: 101, 102, 103, 104
    fold_id 4 → test: 103, 104 | val: 105, 106 | train: 107, 108, 101, 102
    '''
    splits_pam = {
        1: {"test": ["101", "102"], "val": ["103", "104"], "train": ["105", "106", "107", "108"]},
        2: {"test": ["107", "108"], "val": ["101", "102"], "train": ["103", "104", "105", "106"]},
        3: {"test": ["105", "106"], "val": ["107", "108"], "train": ["101", "102", "103", "104"]},
        4: {"test": ["103", "104"], "val": ["105", "106"], "train": ["107", "108", "101", "102"]}
    }ONO

    split = splits_pam[fold_id]
    base_path = '../DATASETS/PAMAP2_Dataset/Protocol/'

    training_files = [base_path + 'subject' + subject + '.dat' for subject in split["train"]]
    validation_files = [base_path + 'subject' + subject + '.dat' for subject in split["val"]]
    test_files = [base_path + 'subject' + subject + '.dat' for subject in split["test"]]

    return training_files, validation_files, test_files

In [5]:
training_files, validation_files, test_files = data_split_PAM(1)

In [6]:
def load_PAM_loco_data(training_files, validation_files, test_files, verbose = False) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    '''
    Load PAMAP2 dataset and process all the data for the model
    '''
    training_data = load_pamap2(training_files, add_group_id = True)
    validation_data = load_pamap2(validation_files, add_group_id = True)
    test_data = load_pamap2(test_files, add_group_id = True)
    
    # ----- Selection of the Columns -----
    training_data_selected = filter_loco_pam(training_data)
    validation_data_selected = filter_loco_pam(validation_data)
    test_data_selected = filter_loco_pam(test_data)
    gc.collect()

    # ----- Clean label 0 ------ #MOD: REMOVE ALL ROWS WITH LABEL 0. 
    training_data_nan = remove_zero_label_rows(training_data_selected)
    validation_data_nan = remove_zero_label_rows(validation_data_selected)
    test_data_nan = remove_zero_label_rows(test_data_selected)
    if verbose:
        print(f"{'-'*90}")
        print(f"Sensor subset training shape: {training_data_nan.shape}\nSensor Validation training shape: {validation_data_nan.shape}\nSensor Test training shape: {test_data_nan.shape}")  
    gc.collect()
    
     # ----- For verification if there are rows with any NaN value  ----- 
    training_nan = incomplete_labeled_rows(training_data_nan)
    validation_nan = incomplete_labeled_rows(validation_data_nan)
    test_nan = incomplete_labeled_rows(test_data_nan)
    if verbose:
        print(f"{'-'*70}")
        print(f"Rows containing NaNs - training: {training_nan}\nRows containing NaNs - validation: {validation_nan}\nRows containing NaNs - test: {test_nan}")  
        
    # ----- Reset index ----- 
    training_data_nan = training_data_nan.reset_index(drop=True)
    validation_data_nan = validation_data_nan.reset_index(drop=True)
    test_data_nan = test_data_nan.reset_index(drop=True) 
    if verbose:
        print(f"{'-'*90}")
        print(f"Reindexed training shape: {training_data_nan.shape}\nReindexed validation shape: {validation_data_nan.shape}\nReindexed test shape: {test_data_nan.shape}")
   
    # ----- Interpolation -----
    training_data_cleaned = interpolation_pam(training_data_nan, max_gap=30)
    validation_data_cleaned = interpolation_pam(validation_data_nan, max_gap=30)
    test_data_cleaned = interpolation_pam(test_data_nan, max_gap=30)      
    if verbose:
        print(f"{'-'*90}")
        print(f"After Interpolation. Training shape: {training_data_cleaned.shape}\nAfter Interpolation. Validation shape: {validation_data_cleaned.shape}\nAfter Interpolation. Test shape: {test_data_cleaned.shape}")   
    gc.collect() 
    
# --------------------------------------------------
# 1) physics-based scaling
# --------------------------------------------------
    if verbose: 
        print("\n")
        print("Descriptive statistics for ACC channels (DEVICE_1) - training set")
        print(training_data_cleaned.iloc[:, 0:3].describe())
        print("\nDescriptive statistics for GYRO channels (DEVICE_1) - training set")
        print(training_data_cleaned.iloc[:, 3:6].describe())
        print("\nDescriptive statistics for MAG channels (DEVICE_1) - training set")
        print(training_data_cleaned.iloc[:, 6:9].describe())
        print(f"{'-'*90}")
    
    # SCALING ACC
    training_data_scaled = acc_data_scaling_pam(training_data_cleaned)
    validation_data_scaled = acc_data_scaling_pam(validation_data_cleaned)
    test_data_scaled = acc_data_scaling_pam(test_data_cleaned)
    
    # NORM MAG               
    training_data_scaled = mag_data_norm_pam(training_data_scaled)
    validation_data_scaled = mag_data_norm_pam(validation_data_scaled)
    test_data_scaled = mag_data_norm_pam(test_data_scaled)
    
    if verbose:
        print("\n")
        print("Descriptive statistics for ACC channels (DEVICE_1) - training set")
        print(training_data_scaled.iloc[:, 0:3].describe())
        print("\nDescriptive statistics for scaled GYRO channels (DEVICE_1) - training set")
        print(training_data_scaled.iloc[:, 3:6].describe())
        print("\nDescriptive statistics for MAG channels (DEVICE_1) - training set")
        print(training_data_scaled.iloc[:, 6:9].describe())

# --------------------------------------------------
# 2) columns to standardize ACC AND GYRO
# --------------------------------------------------    
    scaler = StandardScaler()
    acc_gyro_cols = [axis + offset for offset in range(0, 27, 9) for axis in range(6)]
    
    training_data_scaled.iloc[:, acc_gyro_cols] = scaler.fit_transform(training_data_scaled.iloc[:, acc_gyro_cols].values)
    validation_data_scaled.iloc[:, acc_gyro_cols] = scaler.transform(validation_data_scaled.iloc[:, acc_gyro_cols].values)
    test_data_scaled.iloc[:, acc_gyro_cols] = scaler.transform(test_data_scaled.iloc[:, acc_gyro_cols].values)
    ############################################
    # Stratify
    combined_eval = pd.concat([validation_data_scaled, test_data_scaled], axis = 0, ignore_index = True) 
    y = combined_eval.iloc[:, -2].values       # Labels
    groups = combined_eval["group_id"].values  # File id Name
        
    sgkf = StratifiedGroupKFold(n_splits = 2, shuffle = True, random_state = 42)
    val_idx, test_idx = next(sgkf.split(combined_eval, y, groups))
    print(f"{'-'*20}STRATIY VAL/TEST{'-'*20}")
    print("Validation GROUPS")
    print(combined_eval.iloc[val_idx]["group_id"].value_counts().sort_index())
    print("Test GROUPS")
    print(combined_eval.iloc[test_idx]["group_id"].value_counts().sort_index(), "\n")
    
    #val_with_groups = combined_eval.iloc[val_idx].copy()
    #test_with_groups = combined_eval.iloc[test_idx].copy()
    #print("Validation Split label proportion within each group")
    #print(pd.crosstab(val_with_groups["group_id"], val_with_groups.iloc[:, -2], normalize="index"))
    #print("Test Split label proportion within each group")
    #print(pd.crosstab(test_with_groups["group_id"], test_with_groups.iloc[:, -2], normalize="index"))
    
    new_validation_data_scaled = combined_eval.iloc[val_idx].copy()
    new_test_data_scaled = combined_eval.iloc[test_idx].copy()
    print(f"\n Validation Split Label Proportion")
    print(new_validation_data_scaled.iloc[:, -2].value_counts(normalize=True).sort_index())
    print("Test Split Label Proportion")
    print(new_test_data_scaled.iloc[:, -2].value_counts(normalize=True).sort_index())
    ############################################

    # ----- Sliding Windows -----
    X_windows, y_windows = create_pamap_windows_from_split(training_data_scaled, group_id = "group_id", window_size = 500, stride = 250, target_len = 150, resample_method = "median")
    X_validation_windows, y_validation_windows = create_pamap_windows_from_split(new_validation_data_scaled, group_id = "group_id", window_size = 500, stride = 250, target_len = 150, resample_method = "median")
    X_test_windows, y_test_windows = create_pamap_windows_from_split(new_test_data_scaled, group_id = "group_id", window_size = 500, stride = 250, target_len = 150, resample_method = "median")

    #MAG per window demeaning
    X_windows = mag_data_rotation_pam(X_windows)
    X_validation_windows = mag_data_rotation_pam(X_validation_windows)
    X_test_windows = mag_data_rotation_pam(X_test_windows)
    if verbose:
        print(f"{'-'*90}")
        print(f"Training (windows): {X_windows.shape}. Training Labels {y_windows.shape}")
        print(f"Validation (windows): {X_validation_windows.shape}. Validation Labels {y_validation_windows.shape}")
        print(f"Test (windows): {X_test_windows.shape}. Test Labels {y_test_windows.shape}")
    return X_windows, y_windows, X_validation_windows, y_validation_windows, X_test_windows, y_test_windows

    

In [7]:
X_windows, y_windows, X_validation_windows, y_validation_windows, X_test_windows, y_test_windows = load_PAM_loco_data(training_files, validation_files, test_files, verbose = False)

--------------------STRATIY VAL/TEST--------------------
Validation GROUPS
group_id
102    103884
104    105177
Name: count, dtype: int64
Test GROUPS
group_id
101     94637
103    100374
Name: count, dtype: int64 


 Validation Split Label Proportion
27
1.0    0.222313
2.0    0.228818
3.0    0.240514
4.0    0.308355
Name: proportion, dtype: float64
Test Split Label Proportion
27
1.0    0.252452
2.0    0.267887
3.0    0.216654
4.0    0.263006
Name: proportion, dtype: float64


In [8]:
def make_loaders_PAM(X_windows, y_windows, X_validation_windows, y_validation_windows, X_test_windows, y_test_windows, generator, verbose=False) -> tuple[DataLoader, DataLoader, DataLoader, LabelEncoder]:
    '''
    Creates DataLoaders for training, validation and test sets.
    Called once per seed — generator ensures reproducible shuffling.

    Args:
    X_windows, y_windows: training windows and labels
    X_validation_windows, y_validation_windows: validation windows and labels
    X_test_windows, y_test_windows: test windows and labels
    generator: seeded for reproducibility
    verbose: print class counts and batch shapes
    '''
    # ----- LabelEncoder, Transform, DataLoader -----
    label_encoder = fit_labelencoder(X_windows, y_windows)
    training_dataset = Dataset_HAR(X_windows, y_windows, label_encoder=label_encoder)
    validation_dataset = Dataset_HAR(X_validation_windows, y_validation_windows, label_encoder=label_encoder)
    test_dataset = Dataset_HAR(X_test_windows, y_test_windows, label_encoder=label_encoder)

    train_loader = DataLoader(training_dataset, batch_size = 32, shuffle = True, generator = generator, num_workers = 0) #pin_memory = True, persistent_workers = True)
    val_loader = DataLoader(validation_dataset, batch_size = 32, shuffle = False, num_workers = 0)# pin_memory = True, persistent_workers = True)
    test_loader = DataLoader(test_dataset, batch_size = 32, shuffle = False, num_workers = 0)# pin_memory = True, persistent_workers = True)

    # ----- Samples per label checking and batch size -----
    if verbose:
        label_to_name = {0: "LIE", 1: "SIT", 2: "STAND", 3: "WALK"}
        class_counts = {name: 0 for name in label_to_name.values()}
        for _, y_batch in train_loader:
            for label in y_batch:
                label_idx = label.item()
                class_name = label_to_name[label_idx]
                class_counts[class_name] += 1

        print(f"{'-'*90}")
        print("Training set class distribution:--")
        print(class_counts)
        print(f"{'-'*90}")
    return train_loader, val_loader, test_loader, label_encoder

In [9]:
@torch.no_grad()
def validate_model(model, val_loader, device, criterion):
    '''
    Validation: avg loss per window and accuracy per window.
    '''
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0
    all_preds = []
    all_labels = []
    
    for batch_idx, (x_batch, y_batch) in enumerate(val_loader):
        x_batch, y_batch = x_batch.to(device, non_blocking = True), y_batch.to(device, non_blocking = True)
        logits_ = model(x_batch)                                               # forward Pass
        loss = criterion(logits_, y_batch)                                     # computes mean batch loss

        bsize = y_batch.size(0)
        total_loss += loss.item() * bsize                                      # Total loss contribution of this batch

        predictions = logits_.argmax(dim = -1)                                 # gets the predicted class for each window
        total_correct += (predictions == y_batch).sum().item()                 # counts correct predictions
        total_samples += bsize

        all_preds.extend(predictions.cpu().numpy())
        all_labels.extend(y_batch.cpu().numpy())

        if batch_idx == 0:  # just first batch, so it doesn't spam too much
            true_ids, true_counts = torch.unique(y_batch, return_counts=True)
            pred_ids, pred_counts = torch.unique(predictions, return_counts=True)
    report = classification_report(all_labels, all_preds, target_names = ["LIE", "SIT", "STAND", "WALK"])
    return total_loss / total_samples, total_correct / total_samples, report

In [10]:
#  ----------------------------------------------------------------------------------------------------------------------
#  ---------------------------------------------------- LOSO TRAINING ---------------------------------------------------
for seed in [42, 58, 7, 128, 92]: # [42, 58, 7, 128, 92]
    g = set_seed(seed)
    train_loader, val_loader, test_loader, label_encoder = make_loaders_PAM(X_windows, y_windows, X_validation_windows, y_validation_windows, X_test_windows, y_test_windows, generator = g, verbose = True)
    
    #  ----------- TRAINING SETUP -----------
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    num_classes = int(len(label_encoder.classes_))
    pam_config = HARMambaConfig(num_sensor_features=X_windows.shape[2])
    model = MambaClassificationModel(config = pam_config, num_classes = num_classes).to(device, non_blocking = True) 
    
    # ----------------------
    num_epochs = 12
    lr = 0.00001
    patience = 4
    #WCE
    y_train_encoded = label_encoder.transform(np.asarray(y_windows))
    counts = np.bincount(y_train_encoded, minlength=num_classes)
    weights = 1.0 / np.sqrt(counts)
    weights = torch.tensor(weights, dtype=torch.float32, device=device)
    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = torch.optim.AdamW(model.parameters(), lr = lr, weight_decay = 5e-4)
    #  ----------- TRAINING -----------
    model_name = f"models_pt/model_OPP_fold{1}_seed{seed}.pt"
    epoch_history = []
    best_val_loss = float("inf")
    best_val_acc = 0.0
    best_epoch = None
    best_state = None
    bad_epochs = 0
    with open(f"logs/training_OPP_fold{1}.txt", "a") as log_file:
        log_file.write(f"\nTRAINING STARTING AT: {datetime.now()}\n")
        log_file.write(f"Model: {model_name} | SEED: {seed}\n")
        log_file.flush()
        for epoch in range(num_epochs):
            epoch_start = time.time() # START EPOCH TIME
            model.train()
            total_loss = 0.0
            total_correct = 0
            total_samples = 0
    
            loop = tqdm(train_loader, desc= f"Epoch {epoch+1}/{num_epochs}")
            for batch_idx, (x_batch, y_batch) in enumerate(loop):
                x_batch, y_batch = x_batch.to(device, non_blocking = True), y_batch.to(device, non_blocking = True)    
                optimizer.zero_grad()                                             # clear previous gradients
                logits_ = model(x_batch)                                          # forward pass
                loss = criterion(logits_, y_batch)                                # computes mean batch loss
                loss.backward()                                                   # backward pass: compute gradients
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # prevent exploding gradients
                optimizer.step() 
   
                bsize = y_batch.size(0)
                total_loss += loss.item() * bsize                                 # Total loss contribution of this batch
                
                predictions = logits_.argmax(dim = -1)                            # gets the predicted class for each window: picks the highest score in the last dimension (one highest score per window among the 4 classes)
                total_correct += (predictions == y_batch).sum().item()            # number of correct predictions
                total_samples += bsize
                loop.set_postfix(loss= f"{total_loss/total_samples:.4f}", acc=f"{total_correct/total_samples:.4f}")
    
            train_loss, train_acc = total_loss / total_samples, total_correct / total_samples
            val_loss, val_acc, report = validate_model(model, val_loader, device, criterion)
            if device.type == "cuda":
                torch.cuda.synchronize()
            epoch_time = time.time() - epoch_start # END EPOCH TIME
            epoch_history.append({
                "epoch": epoch + 1,
                "tr_loss": float(train_loss),
                "tr_acc": float(train_acc),
                "val_loss": float(val_loss),
                "val_acc": float(val_acc)                
            })
            print(f"\nEpoch: {epoch+1}/{num_epochs} | tr_Loss: {train_loss:.4f} | tr_acc: {train_acc:.4f} | val_loss: {val_loss:.4f} | val_acc: {val_acc:.4f} | epoch_time: {epoch_time:.2f}s")   
            log_file.write(f"Epoch: {epoch+1}/{num_epochs} | tr_loss: {train_loss:.4f} | tr_acc: {train_acc:.4f} | val_loss: {val_loss:.4f} | val_acc: {val_acc:.4f} | epoch_time: {epoch_time:.2f}s\n")
            log_file.flush()   
            if val_loss < best_val_loss - 1e-12:
                best_val_loss = float(val_loss)
                best_val_acc = float(val_acc)
                best_epoch = epoch + 1                   
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                bad_epochs = 0
            else:
                bad_epochs += 1
                if bad_epochs >= patience:
                    print(f"\nEarly stopping at epoch {epoch + 1} | Best Validation Loss: {best_val_loss:.4f}")
                    break
                    
        # Load Best Model !            
        if best_state is not None:
            model.load_state_dict(best_state)
            torch.save(best_state, model_name)
            # Run val on best model to generate report
            _, _, report = validate_model(model, val_loader, device, criterion) 
            print(report)
            log_file.write(f"\nBest Model Validation Report: \n{str(report)}")        
        log_file.write(f"TRAINING ENDING AT: {datetime.now()}\n")                           
        # ----------------------        
        acc, report, f1, conf_matrix = test_model(model, test_loader, device)
        seed_result = {
            "seed": int(seed),
            "fold": int(1),
            "history": epoch_history,
            "summary": {
                "best_epoch": best_epoch,
                "best_val_loss": float(best_val_loss),
                "best_val_acc": float(best_val_acc),
                "test_accuracy": float(acc),
                "test_report": report,
                "test_f1": float(f1),
                "test_conf_matrix": conf_matrix.tolist()               
            }
        }
        save_json("OPP", "1", seed, seed_result)
        print(f"{'-'*90}")
        print(f"Test Results:\n Accuracy: {acc}\n Report:\n {report}\n F1: {f1}\n Confusion Matrix:\n {conf_matrix}")
        log_file.write(f"\nTest Results:\n Accuracy: {acc}\n Report:\n {report}\n F1: {f1}\n Confusion Matrix:\n {conf_matrix}")
#  ----------------------------------------------------------------------------------------------------------------------

------------------------------------------------------------------------------------------
Training set class distribution:--
{'LIE': 385, 'SIT': 341, 'STAND': 388, 'WALK': 489}
------------------------------------------------------------------------------------------


Epoch 1/12: 100%|██████████| 51/51 [00:20<00:00,  2.49it/s, acc=0.6800, loss=1.1640]



Epoch: 1/12 | tr_Loss: 1.1640 | tr_acc: 0.6800 | val_loss: 1.0300 | val_acc: 0.9412 | epoch_time: 20.82s


Epoch 2/12: 100%|██████████| 51/51 [00:01<00:00, 30.88it/s, acc=0.9058, loss=0.8162]



Epoch: 2/12 | tr_Loss: 0.8162 | tr_acc: 0.9058 | val_loss: 0.7936 | val_acc: 0.9376 | epoch_time: 1.91s


...


Epoch: 10/12 | tr_Loss: 0.2013 | tr_acc: 0.9644 | val_loss: 0.5360 | val_acc: 0.8247 | epoch_time: 1.91s


Epoch 11/12: 100%|██████████| 51/51 [00:01<00:00, 30.84it/s, acc=0.9644, loss=0.1867]



Epoch: 11/12 | tr_Loss: 0.1867 | tr_acc: 0.9644 | val_loss: 0.5370 | val_acc: 0.8187 | epoch_time: 1.91s


Epoch 12/12: 100%|██████████| 51/51 [00:01<00:00, 30.84it/s, acc=0.9707, loss=0.1675]



Epoch: 12/12 | tr_Loss: 0.1675 | tr_acc: 0.9707 | val_loss: 0.6074 | val_acc: 0.8127 | epoch_time: 1.91s

Early stopping at epoch 12 | Best Validation Loss: 0.4577
              precision    recall  f1-score   support

         LIE       1.00      0.95      0.97       185
         SIT       0.97      0.47      0.64       192
       STAND       0.64      0.98      0.77       200
        WALK       0.98      0.98      0.98       256

    accuracy                           0.86       833
   macro avg       0.90      0.85      0.84       833
weighted avg       0.90      0.86      0.85       833

------------------------------------------------------------------------------------------
Test Results:
 Accuracy: 0.767052767052767
 Report:
 {'LIE': {'precision': 1.0, 'recall': 0.826530612244898, 'f1-score': 0.9050279329608939, 'support': 196.0}, 'SIT': {'precision': 0.7266187050359713, 'recall': 0.48325358851674644, 'f1-score': 0.5804597701149425, 'support': 209.0}, 'STAND': {'precision': 0.5